<a href="https://colab.research.google.com/github/rodrix745/taller_patrones_de_dise-o/blob/main/Taller_Patrones.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Integrantes:**
---

Luis Rodrigo Rivera

Juan David Peña



# **1. PATRONES CREACIONALES**

**1.1. Builder – Construir objetos paso a paso**

**Problema:** queremos crear cafés personalizados (tamaño, leche, azúcar, extras). Muchos parámetros.

Creamos la clase compleja Coffee:

In [ ]:
class Coffee:
  def __init__(self, size=None, milk=None, sugar=0, extras=None):
    self.size = size
    self.milk = milk
    self.sugar = sugar
    self.extras = extras or []
  def __str__(self):
    return f"Coffee(size={self.size}, milk={self.milk}, sugar={self.sugar}, extras={self.extras})"

Creamos el Builder:

In [ ]:
class CoffeeBuilder:
  def __init__(self):
    self.reset()
  def reset(self):
    self._size = None
    self._milk = None
    self._sugar = 0
    self._extras = []
    return self

  def set_size(self, size):
    self._size = size
    return self

  def add_milk(self, milk_type):
    self._milk = milk_type
    return self

  def add_sugar(self, spoons):
    self._sugar = spoons
    return self

  def add_extra(self, extra):
    self._extras.append(extra)
    return self

  def build(self):
    coffee = Coffee(self._size, self._milk, self._sugar, self._extras)
    self.reset()
    return coffee

Usarlo:

In [ ]:
builder = CoffeeBuilder()
latte = (builder

.set_size("grande")
.add_milk("almendra")
.add_sugar(2)
.add_extra("canela")
.build())
espresso = builder.set_size("pequeño").build()
print(latte)
print(espresso)

Coffee(size=grande, milk=almendra, sugar=2, extras=['canela'])
Coffee(size=pequeño, milk=None, sugar=0, extras=[])


**1.2. Prototype – Clonar objetos**

**Problema:** tenemos una plantilla de promoción y queremos copiarla varias veces y luego cambiar pequeños detalles.

Creamos la clase con método clone:

In [ ]:
import copy
class PromoBanner:
  def __init__(self, title, text, discount):
    self.title = title
    self.text = text
    self.discount = discount

  def clone(self):
    return copy.deepcopy(self)
  def __str__(self):
    return f"[{self.title}] {self.text} ({self.discount}% off)"

Creamos un “prototipo” y lo clonamos:

In [ ]:
base_promo = PromoBanner(
title="Promo de la semana",
text="Café + croissant",
discount=20
)

promo_lunes = base_promo.clone()
promo_lunes.title = "Lunes de energía"
promo_viernes = base_promo.clone()
promo_viernes.discount = 30
print(base_promo)
print(promo_lunes)
print(promo_viernes)

[Promo de la semana] Café + croissant (20% off)
[Lunes de energía] Café + croissant (20% off)
[Promo de la semana] Café + croissant (30% off)


**1.3. Singleton – Una sola instancia**

**Problema:** queremos una sola instancia de AppConfig (configuración global).

Creamos la clase:

In [ ]:
class AppConfig:
  _instance = None
  def __new__(cls, *args, **kwargs):
    if not cls._instance:
      cls._instance = super().__new__(cls)
    return cls._instance
  def __init__(self, db_url=None):
    if not hasattr(self, "db_url"):
      self.db_url = db_url or "sqlite://local.db"

La usamos:

In [ ]:
config1 = AppConfig("postgres://prod")
config2 = AppConfig("otro") # no cambia, sigue siendo el primero
print(config1.db_url) # postgres://prod
print(config2.db_url) # postgres://prod
print(config1 is config2) # True

postgres://prod
postgres://prod
True


**1.4. Factory Method – Fábrica de objetos**

**Problema:** queremos crear distintos tipos de chocolates. Tú solo haces el pedido y **la fábrica** te entrega el chocolate correcto.

Crear clases por tipo de chocolate:

In [ ]:
class Chocolate:
  def prepare(self):
    raise NotImplementedError

class ChocolateBar(Chocolate):
  def prepare(self):
    return "Preparando una barra de chocolate Wonka"

class FizzyLiftingDrink(Chocolate):
  def prepare(self):
    return "Preparando un Fizzy Lifting Drink (bebida burbujeante)"

class EverlastingGobstopper(Chocolate):
  def prepare(self):
    return "Preparando un Everlasting Gobstopper (caramelo eterno)"

La “Fábrica” Wonka con el método Factory:

In [ ]:
class WonkaFactory:
  def create_candy(self, candy_type: str) -> Chocolate:
    candy_type = candy_type.lower()
    if candy_type == "bar":
      return ChocolateBar()
    elif candy_type == "fizzy":
      return FizzyLiftingDrink()
    elif candy_type == "gobstopper":
      return EverlastingGobstopper()
    else:
      raise ValueError("Ese dulce no existe en la fábrica de Willy Wonka.")

Usarla:

In [ ]:
factory = WonkaFactory()
print(factory.create_candy("bar").prepare())
print(factory.create_candy("fizzy").prepare())
print(factory.create_candy("gobstopper").prepare())

Preparando una barra de chocolate Wonka
Preparando un Fizzy Lifting Drink (bebida burbujeante)
Preparando un Everlasting Gobstopper (caramelo eterno)


# **2. PATRONES ESTRUCTURALES**

**2.1. Adapter – Adaptar una interfaz a otra**

**Problema:** tenemos un sistema nuevo que espera un método *pay(amount)*, pero la pasarela de pagos antigua expone *make_payment(value_in_cents)*. Queremos reutilizar la pasarela vieja sin modificarla.

Definimos la interfaz objetivo y el sistema legacy:

In [ ]:
# Interfaz que espera nuestra aplicación nueva
class PaymentProcessor:
  def pay(self, amount: float) -> str:
    raise NotImplementedError
# Sistema legacy (no lo podemos cambiar)
class LegacyPaymentGateway:
  def make_payment(self, value_in_cents: int) -> str:
    return f"Pago legacy procesado por {value_in_cents} centavos"

Creamos el Adapter:

In [ ]:
class LegacyPaymentAdapter(PaymentProcessor):
  def __init__(self, legacy_gateway: LegacyPaymentGateway):
    self.legacy = legacy_gateway
  def pay(self, amount: float) -> str:
  # Adapta la unidad (dólares -> centavos) y el nombre del método
    cents = int(amount * 100)
    return self.legacy.make_payment(cents)

Usarlo:

In [ ]:
gateway = LegacyPaymentGateway()
processor = LegacyPaymentAdapter(gateway)
print(processor.pay(25.50)) # Pago legacy procesado por 2550 centavos

Pago legacy procesado por 2550 centavos


**2.2. Decorator – Añadir funcionalidades dinámicamente**

**Problema:** tenemos un sistema de notificaciones. A veces solo queremos enviar correo, a veces correo + SMS, a veces correo + SMS + Slack. No queremos crear una clase distinta por cada combinación.

Componente base y notificación concreta:

In [ ]:
class Notifier:
  def send(self, message: str) -> None:
    raise NotImplementedError
class EmailNotifier(Notifier):
  def send(self, message: str) -> None:
    print(f"[Email] {message}")

Decoradores que envuelven a otro Notifier:

In [ ]:
class NotifierDecorator(Notifier):
  def __init__(self, wrapped: Notifier):
    self._wrapped = wrapped
  def send(self, message: str) -> None:
    self._wrapped.send(message)
class SMSDecorator(NotifierDecorator):
  def send(self, message: str) -> None:
    super().send(message)
    print(f"[SMS] {message}")
class SlackDecorator(NotifierDecorator):
  def send(self, message: str) -> None:
    super().send(message)
    print(f"[Slack] {message}")

Componer en tiempo de ejecución:

In [ ]:
notifier = SlackDecorator(SMSDecorator(EmailNotifier()))
notifier.send("Servidor caído")

[Email] Servidor caído
[SMS] Servidor caído
[Slack] Servidor caído


**2.3. Facade – Una fachada simple sobre un subsistema complejo**

**Problema:** para ver una película en casa hay que prender el proyector, bajar la pantalla, encender el sonido y poner el reproductor. Queremos un solo botón: *ver_pelicula()*.

El subsistema (varias clases independientes):

In [ ]:
class Projector:
  def on(self): print("Proyector encendido")
  def off(self): print("Proyector apagado")

class SoundSystem:
  def set_volume(self, level): print(f"Volumen en {level}")
  def off(self): print("Sonido apagado")

class StreamingPlayer:
  def play(self, movie): print(f"Reproduciendo '{movie}'")
  def stop(self): print("Reproducción detenida")

La fachada que lo orquesta todo:

In [ ]:
class HomeTheaterFacade:
  def __init__(self, projector, sound, player):
    self.projector = projector
    self.sound = sound
    self.player = player
  def ver_pelicula(self, movie):
    self.projector.on()
    self.sound.set_volume(7)
    self.player.play(movie)
  def terminar(self):
    self.player.stop()
    self.sound.off()
    self.projector.off()

Usarla:

In [ ]:
theater = HomeTheaterFacade(Projector(), SoundSystem(), StreamingPlayer())
theater.ver_pelicula("Interstellar")
theater.terminar()

Proyector encendido
Volumen en 7
Reproduciendo 'Interstellar'
Reproducción detenida
Sonido apagado
Proyector apagado


**2.4. Composite – Tratar objetos individuales y grupos de la misma forma**

**Problema:** queremos calcular el tamaño total de una carpeta del sistema de archivos. Una carpeta puede contener archivos y otras carpetas, recursivamente.

Componente común (hojas y compuestos comparten interfaz):

In [ ]:
class FileSystemItem:
  def __init__(self, name: str):
    self.name = name

  def get_size(self) -> int:
    raise NotImplementedError

Hoja (File) y Compuesto (Folder):

In [ ]:
class File(FileSystemItem):
  def __init__(self, name: str, size: int):
    super().__init__(name)
    self._size = size
  def get_size(self) -> int:
    return self._size
class Folder(FileSystemItem):
  def __init__(self, name: str):
    super().__init__(name)
    self.children: list[FileSystemItem] = []
  def add(self, item: FileSystemItem):
    self.children.append(item)
  def get_size(self) -> int:
    return sum(child.get_size() for child in self.children)

Construir el árbol y consultarlo:

In [ ]:
root = Folder("proyecto")
root.add(File("README.md", 2))
src = Folder("src")
src.add(File("main.py", 10))
src.add(File("utils.py", 5))
root.add(src)
print(root.get_size()) # 17 (se trata igual un archivo que una carpeta)

17


# **3. PATRONES DE COMPORTAMIENTO**

**3.1. Strategy – Algoritmos intercambiables**

**Problema:** una tienda aplica distintos descuentos: cliente normal (0%), estudiante (10%), empleado (20%) y VIP (30%). Queremos cambiar la regla sin llenar el código de *if/elif*.

Estrategias intercambiables:

In [ ]:
class DiscountStrategy:
  def apply(self, total: float) -> float:
    raise NotImplementedError
class NoDiscount(DiscountStrategy):
  def apply(self, total): return total
class StudentDiscount(DiscountStrategy):
  def apply(self, total): return total * 0.90
class EmployeeDiscount(DiscountStrategy):
  def apply(self, total): return total * 0.80
class VIPDiscount(DiscountStrategy):
  def apply(self, total): return total * 0.70

Contexto que recibe la estrategia:

In [ ]:
class Cart:
  def __init__(self, strategy: DiscountStrategy):
    self.items = []
    self.strategy = strategy
  def add(self, price: float):
    self.items.append(price)
  def total(self) -> float:
    return self.strategy.apply(sum(self.items))

Cambiar la estrategia sin tocar el carrito:

In [ ]:
cart = Cart(StudentDiscount())
cart.add(100); cart.add(50)
print(cart.total()) # 135.0
cart.strategy = VIPDiscount()
print(cart.total()) # 105.0

135.0
105.0


**3.2. Observer – Notificar a varios suscriptores**

**Problema:** cuando un canal de YouTube publica un video, todos sus suscriptores deben enterarse. El canal no debería conocer a cada suscriptor por nombre.

Sujeto (observable) y observador:

In [ ]:
class Subscriber:
  def update(self, video: str):
    raise NotImplementedError
class Channel:
  def __init__(self, name: str):
    self.name = name
    self._subs: list[Subscriber] = []
  def subscribe(self, sub: Subscriber):
    self._subs.append(sub)
  def unsubscribe(self, sub: Subscriber):
    self._subs.remove(sub)
  def publish(self, video: str):
    print(f"\n{self.name} publicó: {video}")
    for sub in self._subs:
      sub.update(video)

Suscriptores concretos:

In [ ]:
class EmailSubscriber(Subscriber):
  def __init__(self, email): self.email = email
  def update(self, video):
    print(f" -> Email a {self.email}: nuevo video '{video}'")

class MobileSubscriber(Subscriber):
  def __init__(self, user): self.user = user
  def update(self, video):
    print(f" -> Push a {self.user}: '{video}'")

Usarlo:

In [ ]:
channel = Channel("DevTalks")
channel.subscribe(EmailSubscriber("ana@x.com"))
channel.subscribe(MobileSubscriber("@carlos"))
channel.publish("Patrones de Diseño en Python")


DevTalks publicó: Patrones de Diseño en Python
 -> Email a ana@x.com: nuevo video 'Patrones de Diseño en Python'
 -> Push a @carlos: 'Patrones de Diseño en Python'


**3.3. Command – Encapsular una petición como objeto**

**Problema:** un control remoto debe poder ejecutar acciones (prender luz, apagar luz) y además poder *deshacer* la última acción. Necesitamos representar cada acción como algo manipulable.

Receptor y comandos:

In [ ]:
class Light:
  def __init__(self): self.is_on = False
  def turn_on(self): self.is_on = True; print("Luz: ON")
  def turn_off(self): self.is_on = False; print("Luz: OFF")
class Command:
  def execute(self): raise NotImplementedError
  def undo(self): raise NotImplementedError
class TurnOnCommand(Command):
  def __init__(self, light): self.light = light
  def execute(self): self.light.turn_on()
  def undo(self): self.light.turn_off()
class TurnOffCommand(Command):
  def __init__(self, light): self.light = light
  def execute(self): self.light.turn_off()
  def undo(self): self.light.turn_on()

Invocador (el control remoto) con historial:

In [ ]:
class RemoteControl:
  def __init__(self): self.history: list[Command] = []
  def press(self, command: Command):
    command.execute()
    self.history.append(command)
  def undo(self):
    if self.history:
      self.history.pop().undo()

Usarlo:

In [ ]:
light = Light()
remote = RemoteControl()
remote.press(TurnOnCommand(light)) # Luz: ON
remote.press(TurnOffCommand(light)) # Luz: OFF
remote.undo() # Luz: ON
remote.undo() # Luz: OFF

Luz: ON
Luz: OFF
Luz: ON
Luz: OFF


**3.4. Template Method – Esqueleto de un algoritmo**

**Problema:** queremos generar reportes en distintos formatos (texto plano, HTML). Los pasos son siempre los mismos (encabezado, cuerpo, pie) pero el formato cambia. Queremos fijar el orden y dejar que las subclases personalicen cada paso.

Clase base con el algoritmo completo:

In [ ]:
class Report:
  # Template method: define el orden, no se sobreescribe
  def generate(self, data: dict) -> str:
    parts = [self.header(data), self.body(data), self.footer(data)]
    return "\n".join(parts)
  def header(self, data): raise NotImplementedError
  def body(self, data): raise NotImplementedError
  def footer(self, data): raise NotImplementedError

Subclases que rellenan los “huecos”:

In [ ]:
class TextReport(Report):

  def header(self, data): return f"=== {data['title']} ==="
  def body(self, data): return data['content']
  def footer(self, data): return f"-- Generado por {data['author']} --"
class HTMLReport(Report):
  def header(self, data): return f"<h1>{data['title']}</h1>"
  def body(self, data): return f"<p>{data['content']}</p>"
  def footer(self, data): return f"<footer>{data['author']}</footer>"

Usarlo:

In [ ]:
data = {"title": "Ventas Q4", "content": "Crecimos 12%", "author": "C. Sierra"}
print(TextReport().generate(data))
print()
print(HTMLReport().generate(data))

=== Ventas Q4 ===
Crecimos 12%
-- Generado por C. Sierra --

<h1>Ventas Q4</h1>
<p>Crecimos 12%</p>
<footer>C. Sierra</footer>


**3.5. State – Cambiar de comportamiento según el estado interno**

**Problema:** un pedido en un e-commerce pasa por estados: *Pendiente → Pagado → Enviado → Entregado*. Las acciones permitidas dependen del estado actual. Queremos evitar un *if* gigantesco.

Una clase por estado:

In [ ]:
class OrderState:
  def pay(self, order): print("Acción no permitida en este estado")
  def ship(self, order): print("Acción no permitida en este estado")
  def deliver(self, order):print("Acción no permitida en este estado")
class Pending(OrderState):
  def pay(self, order):
    print("Pago recibido")
    order.state = Paid()
class Paid(OrderState):
  def ship(self, order):
    print("Pedido enviado")
    order.state = Shipped()
class Shipped(OrderState):

  def deliver(self, order):
    print("Pedido entregado")
    order.state = Delivered()
class Delivered(OrderState):
  pass # estado final

Contexto que delega en el estado:

In [ ]:
class Order:
  def __init__(self):
    self.state: OrderState = Pending()
  def pay(self): self.state.pay(self)
  def ship(self): self.state.ship(self)
  def deliver(self): self.state.deliver(self)

Usarlo:

In [ ]:
order = Order()
order.ship() # Acción no permitida en este estado (aún está Pending)
order.pay() # Pago recibido
order.ship() # Pedido enviado
order.deliver() # Pedido entregado

Acción no permitida en este estado
Pago recibido
Pedido enviado
Pedido entregado
